# Tạo `subformulae/magma_subform_50.hdf5` cho `msg_simulation` trên Kaggle

File này chỉ cần cho bước 2 của MARASON (`run_scripts/marason/02_sweep_gen_thresh.py`, dò số mảnh, **tuỳ chọn**).
Notebook chạy đúng lệnh thứ hai của `data_scripts/all_assign_subform.sh`, đổi dataset sang `msg_simulation`:

```
python data_scripts/forms/01_assign_subformulae.py --data-dir data/spec_datasets/msg_simulation/ \
  --labels-file data/spec_datasets/msg_simulation/labels.tsv --use-magma --mass-diff-thresh 20 \
  --output-dir magma_subform_50.hdf5 --num-workers <số CPU>
```

`--use-magma` chạy lại bộ phân mảnh MAGMa cho từng phân tử, rồi gán cho mỗi đỉnh của phổ công thức của mảnh khớp
(sai số ≤ 20 ppm). Bước này tốn CPU gần bằng `run_magma.py`, khoảng 24 giờ cho 119,029 phổ với 4 CPU, nên được chia
shard giống lúc chạy MAGMa. Mỗi phổ được xử lý độc lập, nên gộp các shard cho cùng kết quả như chạy một lần.

**Đầu vào:** *Add Input* dataset `giamyy/msg-simulation`. Notebook tự tìm thư mục chứa `labels.tsv` và `spec_files.hdf5`.

| `STAGE` | Làm gì |
|---|---|
| `"test"` | Chạy `TEST_N` phổ đầu để đo thời gian, rồi ước lượng thời gian cho mỗi shard |
| `"shard"` | Chạy phần `SHARD_ID` trong `NUM_SHARDS` phần (phổ thứ `SHARD_ID`, `SHARD_ID + NUM_SHARDS`, …) |
| `"merge"` | Gộp output của các notebook shard thành một `magma_subform_50.hdf5` |

**Cách chạy:**
1. (Nên làm) một notebook `STAGE = "test"` để xem mỗi shard mất bao lâu. Nếu quá 11 giờ thì tăng `NUM_SHARDS`.
2. `NUM_SHARDS` bản sao: `STAGE = "shard"`, `SHARD_ID` = 0 … `NUM_SHARDS - 1`. *Add Input* `msg-simulation`,
   *Accelerator: None*, *Internet: On* → *Save Version → Save & Run All*. Các bản chạy song song được.
3. Notebook `STAGE = "merge"`: *Add Input* output của mọi notebook shard **và** dataset `msg-simulation` → *Save & Run All*.
   Với `EXPORT_FULL = True`, output là `msg_simulation_processed/`: toàn bộ dataset cũ, cộng thêm
   `subformulae/magma_subform_50.hdf5`.
4. Tab Output → *New Dataset* (ví dụ `msg-simulation-v2`). Trên máy train:
   `python scripts/train_marason_msg.py --kaggle-dataset giamyy/msg-simulation-v2`.

In [ ]:
# ===== CẤU HÌNH =====
REPO_URL     = "https://github.com/Giamyy/ms-pred.git"
STAGE        = "test"             # "test" | "shard" | "merge"
NUM_SHARDS   = 4                  # phải giống nhau ở mọi notebook shard và merge
SHARD_ID     = 0                  # STAGE = "shard": 0 .. NUM_SHARDS-1
TEST_N       = 2000               # STAGE = "test": số phổ chạy thử
DATA_INPUT   = None               # thư mục chứa labels.tsv + spec_files.hdf5; None = tự tìm trong /kaggle/input
SHARD_INPUTS = ["/kaggle/input"]  # STAGE = "merge": nơi tìm output của các notebook shard (tìm đệ quy)
MASS_DIFF_THRESH = 20             # ppm, giống all_assign_subform.sh
OUT_NAME     = "magma_subform_50.hdf5"
# STAGE = "merge": True = chép cả dataset đầu vào (Add Input msg-simulation) ra output, đặt file mới vào subformulae/,
# để tạo một dataset đầy đủ trong một lần. False = output chỉ có magma_subform_50.hdf5.
EXPORT_FULL  = True

In [ ]:
import os, subprocess, sys, json, shutil, time
from pathlib import Path
import pandas as pd
import h5py

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
REPO = Path("/tmp/ms-pred")       # ngoài /kaggle/working: repo và .venv không bị lưu vào output
NCPU = os.cpu_count() or 4
assert STAGE in {"test", "shard", "merge"}, STAGE
assert 0 <= SHARD_ID < NUM_SHARDS

def sh(cmd, cwd=None, check=True):
    """Chạy lệnh shell, in output ngay khi có; bấm Interrupt sẽ dừng cả tiến trình con."""
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, executable="/bin/bash", start_new_session=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in p.stdout:
            print(line, end="")
    except KeyboardInterrupt:
        os.killpg(p.pid, 9)
        raise
    if p.wait() != 0 and check:
        raise RuntimeError(f"Lệnh lỗi (exit {p.returncode}): {cmd}")
    return p

print("CPU cores:", NCPU, "| STAGE:", STAGE)

In [ ]:
# ===== 1. Clone repo + cài môi trường (bỏ qua với STAGE = "merge") =====
PY = str(REPO / ".venv" / "bin" / "python")
if STAGE == "merge":
    print("STAGE = merge: chỉ cần h5py, không cài môi trường")
else:
    import urllib.request
    try:
        urllib.request.urlopen("https://pypi.org", timeout=10)
    except Exception as e:
        raise SystemExit(f"Chưa có internet, hãy bật Settings -> Internet -> On.\n{e}")
    if not REPO.exists():
        sh(f"git clone --depth 1 {REPO_URL} {REPO}")
    sh("pip install -q uv")
    # --python 3.11: Python 3.13 của Kaggle không có wheel torch-scatter / torch-sparse dựng sẵn
    for attempt in range(3):
        if sh("uv sync --extra cpu --python 3.11", cwd=REPO, check=False).returncode == 0:
            break
        print(f"uv sync lỗi (lần {attempt + 1}/3), thử lại...")
        time.sleep(10)
    else:
        raise RuntimeError("uv sync thất bại")
    sh(f"{PY} -c \"import sys, ms_pred, rdkit; from ms_pred.magma import fragmentation; "
       f"print('python', sys.version.split()[0], '| rdkit', rdkit.__version__)\"")

In [ ]:
# ===== 2. Tìm dataset và cắt phần cần chạy =====
def find_data():
    if DATA_INPUT:
        return Path(DATA_INPUT)
    cands = [p.parent for p in Path("/kaggle/input").rglob("labels.tsv") if (p.parent / "spec_files.hdf5").exists()]
    cands.sort(key=lambda d: (not d.name.startswith("msg_simulation"), len(d.parts)))
    if not cands:
        raise SystemExit("Không thấy thư mục nào có labels.tsv + spec_files.hdf5 trong /kaggle/input: "
                         "hãy Add Input dataset giamyy/msg-simulation")
    return cands[0]

if STAGE != "merge":
    SRC = find_data()
    print("Dữ liệu:", SRC)
    labels = pd.read_csv(SRC / "labels.tsv", sep="\t")
    if STAGE == "test":
        part = labels.iloc[:TEST_N]
        RUN_DIR = REPO / "data/spec_datasets/msg_simulation_subform_test"
    else:
        part = labels.iloc[SHARD_ID::NUM_SHARDS]
        RUN_DIR = REPO / f"data/spec_datasets/msg_simulation_subform_shard{SHARD_ID}"
    # /kaggle/input chỉ đọc: chép labels + phổ của phần này vào thư mục ghi được (script ghi vào <data-dir>/subformulae)
    shutil.rmtree(RUN_DIR, ignore_errors=True)
    RUN_DIR.mkdir(parents=True)
    part.to_csv(RUN_DIR / "labels.tsv", sep="\t", index=False)
    missing = 0
    with h5py.File(SRC / "spec_files.hdf5", "r") as src, h5py.File(RUN_DIR / "spec_files.hdf5", "w") as dst:
        for spec in part["spec"].astype(str):
            if f"{spec}.ms" in src:
                src.copy(f"{spec}.ms", dst, name=f"{spec}.ms")
            else:
                missing += 1
    print(f"{len(part)} / {len(labels)} phổ -> {RUN_DIR} (thiếu trong spec_files: {missing})")
    if missing:
        raise RuntimeError("Có phổ trong labels.tsv nhưng không có trong spec_files.hdf5")

In [ ]:
# ===== 3. Gán subformula theo MAGMa (lệnh của data_scripts/all_assign_subform.sh) =====
if STAGE != "merge":
    t0 = time.time()
    sh(f"{PY} data_scripts/forms/01_assign_subformulae.py "
       f"--data-dir {RUN_DIR}/ --labels-file {RUN_DIR}/labels.tsv "
       f"--use-magma --mass-diff-thresh {MASS_DIFF_THRESH} "
       f"--output-dir {OUT_NAME} --num-workers {NCPU}", cwd=REPO)
    minutes = (time.time() - t0) / 60
    out_h5 = RUN_DIR / "subformulae" / OUT_NAME
    with h5py.File(out_h5, "r") as f:
        n_out = len(f.keys())
    print(f"{n_out} mục trong {out_h5.name}, {minutes:.1f} phút")

    if STAGE == "test":
        per_spec = minutes / len(part)
        per_shard = per_spec * len(labels) / NUM_SHARDS / 60
        print(f"Ước lượng: {per_spec * 1000:.1f} phút / 1000 phổ; "
              f"{per_spec * len(labels) / 60:.1f} giờ cho {len(labels)} phổ; "
              f"{per_shard:.1f} giờ mỗi shard với NUM_SHARDS = {NUM_SHARDS}")
        if per_shard > 11:
            need = int(per_spec * len(labels) / 60 / 11) + 1
            print(f"Quá giới hạn 12 giờ của Kaggle: đặt NUM_SHARDS >= {need}")
    else:
        out_dir = WORK / f"magma_subform50_shard{SHARD_ID}"
        shutil.rmtree(out_dir, ignore_errors=True)
        out_dir.mkdir(parents=True)
        shutil.copy2(out_h5, out_dir / OUT_NAME)
        (out_dir / "shard.json").write_text(json.dumps({
            "shard_id": SHARD_ID, "num_shards": NUM_SHARDS, "specs": int(len(part)), "entries": n_out,
            "mass_diff_thresh": MASS_DIFF_THRESH, "source_specs": int(len(labels))}))
        print("Output:", out_dir, "-> Save Version (Save & Run All) để lưu")

In [ ]:
# ===== 4. (STAGE = "merge") gộp các shard =====
if STAGE == "merge":
    found = sorted({p.parent for root in SHARD_INPUTS for p in Path(root).glob(f"**/magma_subform50_shard*/shard.json")})
    metas = [json.loads((d / "shard.json").read_text()) for d in found]
    for d, m in zip(found, metas):
        print("Tìm thấy:", d, m)
    if sorted(m["shard_id"] for m in metas) != list(range(NUM_SHARDS)) or any(m["num_shards"] != NUM_SHARDS for m in metas):
        raise RuntimeError(f"Cần đủ {NUM_SHARDS} shard (0..{NUM_SHARDS - 1}), tìm thấy {[m['shard_id'] for m in metas]}")
    if len({(m["mass_diff_thresh"], m["source_specs"]) for m in metas}) != 1:
        raise RuntimeError("Các shard chạy với tham số hoặc dữ liệu khác nhau")
    if sum(m["specs"] for m in metas) != metas[0]["source_specs"]:
        raise RuntimeError("Tổng số phổ của các shard khác số phổ của dataset")

    out = WORK / OUT_NAME
    written, dup = 0, []
    with h5py.File(out, "w") as dst:
        for d in found:
            with h5py.File(d / OUT_NAME, "r") as src:
                for key in src.keys():
                    if key in dst:
                        dup.append(key)
                        continue
                    src.copy(key, dst, name=key)
                    written += 1
    if dup:
        raise RuntimeError(f"{len(dup)} mục có ở nhiều shard, ví dụ {dup[:5]}")
    expected = sum(m["entries"] for m in metas)
    print(f"{out}: {written} mục (kỳ vọng {expected}), {out.stat().st_size / 2**20:.0f} MB")
    assert written == expected

    if EXPORT_FULL:
        # Dataset đầy đủ = dataset đầu vào + file mới; chép (không symlink) vì /kaggle/input không nằm trong output
        src = find_data()
        full = WORK / "msg_simulation_processed"
        shutil.rmtree(full, ignore_errors=True)
        print(f"Chép {src} -> {full}", flush=True)
        shutil.copytree(src, full, symlinks=False)
        (full / "subformulae").mkdir(exist_ok=True)
        shutil.move(str(out), str(full / "subformulae" / OUT_NAME))
        with h5py.File(full / "subformulae" / OUT_NAME, "r") as f:
            assert len(f.keys()) == written
        n_specs = len(pd.read_csv(full / "labels.tsv", sep="\t"))
        if n_specs != metas[0]["source_specs"]:
            raise RuntimeError(f"Dataset đầu vào có {n_specs} phổ, các shard chạy trên {metas[0]['source_specs']} phổ")
        sh(f"du -sh {full}/* {full}/subformulae/*")
        print("Xong. Tab Output -> New Dataset (vd msg-simulation-v2). Trên máy train: "
              "python scripts/train_marason_msg.py --kaggle-dataset <user>/msg-simulation-v2")
    else:
        print("Xong. Tab Output -> New Dataset. Trên máy train, chép file vào "
              "data/spec_datasets/msg_simulation/subformulae/magma_subform_50.hdf5")